# Open VCV — Training trên Kaggle T4 x2

**Cấu hình:** T4 x2 (2×16GB VRAM) · batch=256 (128/GPU) · AMP FP16 · workers=8

### Cài đặt Kaggle:
1. Settings → Accelerator → **GPU T4 x2**
2. Add dataset: `imagenet-object-localization-challenge`
3. Internet: **ON**
4. **Run All**

| Accelerator | Uớc tính/epoch | Ghi chú |
|---|---|---|
| T4 x1 | ~16 phút | AMP FP16 |
| **T4 x2** | **~8 phút** | DataParallel, AMP FP16 |
| TPU v5e-8 | ~3-5 phút | BF16, dùng notebook riêng |


In [ ]:
# ── Cell 1: Clone / update repo ──────────────────────────────────────────────
import os

REPO_URL = 'https://github.com/Bigkatoan/open_vcv.git'
REPO_DIR = '/kaggle/working/open_vcv'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone {REPO_URL} {REPO_DIR}')
else:
    os.system(f'cd {REPO_DIR} && git pull')

os.chdir(REPO_DIR)
print(f'Working dir: {os.getcwd()}')


In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────────
os.system('pip install -q -r requirements.txt 2>/dev/null || '
          'pip install -q torch torchvision tqdm matplotlib pandas scikit-learn')


In [ ]:
# ── Cell 3: Kiểm tra môi trường ──────────────────────────────────────────────
import torch
print(f'torch version : {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {props.name} ({props.total_memory // 1024**3}GB)')


In [ ]:
# ── Cell 4: Cấu hình paths ───────────────────────────────────────────────────
IMAGENET_TRAIN = '/kaggle/input/imagenet-object-localization-challenge/ILSVRC/Data/CLS-LOC/train'

if os.path.exists(IMAGENET_TRAIN):
    class_dirs = [d for d in os.listdir(IMAGENET_TRAIN) if os.path.isdir(os.path.join(IMAGENET_TRAIN, d))]
    print(f'ImageNet OK: {len(class_dirs)} classes found')
else:
    print(f'ERROR: {IMAGENET_TRAIN} not found!')
    print('  -> Add dataset imagenet-object-localization-challenge to notebook')


In [ ]:
# ── Cell 5: Train trên T4 x2 ─────────────────────────────────────────────────
import sys, os, torch
print(f'torch {torch.__version__}')
n_gpus = torch.cuda.device_count()
for i in range(n_gpus):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name} ({p.total_memory//1024**3}GB)')

if '/kaggle/working/open_vcv' not in sys.path:
    sys.path.insert(0, '/kaggle/working/open_vcv')
os.chdir('/kaggle/working/open_vcv')

from src.models.VAE import VAE
from src.losses.losses import IntersectUnionLoss
from src.trainers.trainer import Trainer, TrainConfig
print('Imports OK')

cfg = TrainConfig(
    run_name     = 'gated_vae_imagenet_t4x2',
    dataset      = 'imagenet',
    data_root    = IMAGENET_TRAIN,
    img_size     = 224,
    q            = 2,
    k            = 0,
    batch_size   = 256,       # DataParallel chia deu -> 128/GPU
    grad_accum   = 1,
    num_workers  = min(8, os.cpu_count() or 4),
    prefetch_factor = 4,
    device       = 'cuda',
    use_amp      = True,
    use_vae      = False,
    total_epochs = 100,
    lr           = 3e-4,
    log_iter_every = 100,
)
print(f'Config OK | batch={cfg.batch_size} workers={cfg.num_workers} amp={cfg.use_amp}')

model = VAE(
    s1_out=16, s1_heads=4,  s1_blocks=1,
    s2_out=16, s2_heads=8,  s2_blocks=2,
    s3_out=16, s3_heads=16, s3_blocks=2,
    latent_ch=2048,
    dec_ch3=128, dec_ch2=64, dec_ch1=32,
    dim_inter=1024, dim_unique=1024,
    feat_dim=64, hidden_dim=256,
)
n = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model params: {n:,}')

torch.cuda.empty_cache()
loss_fn = IntersectUnionLoss()
Trainer(model, loss_fn, cfg).train()


In [ ]:
# ── Cell 6: Plot training curves ─────────────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt
import glob

EXP_DIRS = sorted(glob.glob('experiments/gated_vae_imagenet_t4x2*'))
if not EXP_DIRS:
    print('No experiment found yet')
else:
    exp = EXP_DIRS[-1]
    df = pd.read_csv(f'{exp}/losses.csv')
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(df['epoch'], df['train_loss'], label='train')
    axes[0].set_title('Loss'); axes[0].legend()
    if 'metrics.csv' in os.listdir(exp):
        mdf = pd.read_csv(f'{exp}/metrics.csv')
        axes[1].plot(mdf['epoch'], mdf['val_loss'], label='val')
        axes[1].set_title('Val metrics'); axes[1].legend()
    plt.tight_layout()
    plt.show()
    print(f'Experiment: {exp}')


In [ ]:
# ── Cell 7: Save checkpoint ra Kaggle output ─────────────────────────────────
import shutil, glob

OUTPUT_DIR = '/kaggle/working/checkpoints'
os.makedirs(OUTPUT_DIR, exist_ok=True)

EXP_DIRS = sorted(glob.glob('experiments/gated_vae_imagenet_t4x2*'))
if EXP_DIRS:
    exp = EXP_DIRS[-1]
    for ckp in glob.glob(f'{exp}/checkpoints/*.pt'):
        dst = os.path.join(OUTPUT_DIR, os.path.basename(ckp))
        shutil.copy2(ckp, dst)
        print(f'Saved: {dst}')
else:
    print('No checkpoint found')
